<a href="https://colab.research.google.com/github/bru02/epstein/blob/main/notebooks/network_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Network analysis

Build email-derived communication and co-participation networks from `bridge_email_people.parquet`, then compare the structural results to Laszlo Pokorny's 2026 paper/dissertation, *Social Network Analysis of Jeffrey Epstein and Other Elite Offenders' Facilitation Networks Through a Psychopossession Lens*.

Important caveat: this notebook analyzes people linked by released emails. Pokorny's Epstein network is described as a facilitation network built from public sources including court documents, flight logs, investigative journalism, and organizational records. The numbers are therefore comparable as SNA structure, not as a replication of the same underlying network.

In [ ]:
is_colab = "google.colab" in str(get_ipython())
if is_colab:
    get_ipython().system("uv pip install --system 'epstein @ git+https://github.com/bru02/epstein'")

In [ ]:
%matplotlib inline
from collections import Counter
from itertools import combinations
import json
from pathlib import Path
from time import perf_counter
import importlib.util

from IPython.display import display
import matplotlib.pyplot as plt
if importlib.util.find_spec("networkx") is None:
    get_ipython().system("uv pip install networkx")
import networkx as nx
import numpy as np
import polars as pl

Path("outputs").mkdir(exist_ok=True)
started_at = perf_counter()
last_at = started_at

# Set to None for the full linked corpus. Keep a sample while iterating.
sample_documents = 250_000
sample_seed = 7
min_directed_edge_weight = 2
min_copresence_edge_weight = 2
max_people_per_email = 25
centrality_sample_nodes = 400
random_disruption_runs = 100


def log(label):
    global last_at
    now = perf_counter()
    print(f"[network] {label}: phase={now - last_at:.2f}s total={now - started_at:.2f}s", flush=True)
    last_at = now


def write_frame(frame, path):
    frame.write_csv(Path("outputs") / path)

## Load linked people

The graph is built from email-person links. Sampling happens at the email level so each sampled message keeps all linked people.

In [ ]:
data_base = "data" if Path("data/fact_emails.parquet").exists() else "https://github.com/bru02/epstein/raw/main/data"

emails = pl.read_parquet(f"{data_base}/fact_emails.parquet", columns=["email_id", "sent_at", "year", "subject", "text_token_len"])
bridge = pl.read_parquet(f"{data_base}/bridge_email_people.parquet")
people = pl.read_parquet(f"{data_base}/dim_people.parquet")

linked_email_ids = bridge.select("email_id").unique()
analysis_emails = emails.join(linked_email_ids, on="email_id", how="semi")
if sample_documents is None:
    sampled_emails = analysis_emails
else:
    sampled_emails = analysis_emails.sample(n=min(sample_documents, analysis_emails.height), seed=sample_seed)

sampled_ids = sampled_emails.select("email_id")
participants = (
    bridge.join(sampled_ids, on="email_id", how="semi")
    .filter(pl.col("involvement_role").is_in(["sender", "to", "cc", "bcc"]))
    .select("email_id", "person_id", "person_name", "involvement_role")
    .unique()
)

log("loaded linked people")
print(f"linked emails available: {analysis_emails.height:,}")
print(f"sampled linked emails: {sampled_emails.height:,}")
print(f"sampled participant links: {participants.height:,}")
print(f"sampled people: {participants.select(pl.col('person_id').n_unique()).item():,}")

## Directed communication graph

A directed edge runs from each sender to each `to`, `cc`, or `bcc` participant on the same email. Edge weight is the number of sampled emails linking the pair.

In [ ]:
senders = participants.filter(pl.col("involvement_role") == "sender").select(
    "email_id",
    pl.col("person_id").alias("source"),
    pl.col("person_name").alias("source_name"),
)
recipients = participants.filter(pl.col("involvement_role").is_in(["to", "cc", "bcc"])).select(
    "email_id",
    pl.col("person_id").alias("target"),
    pl.col("person_name").alias("target_name"),
    pl.col("involvement_role").alias("target_role"),
)

directed_edges = (
    senders.join(recipients, on="email_id", how="inner")
    .filter(pl.col("source") != pl.col("target"))
    .group_by("source", "source_name", "target", "target_name")
    .agg(
        pl.len().alias("weight"),
        pl.col("email_id").n_unique().alias("emails"),
        pl.col("target_role").unique().str.join(", ").alias("target_roles"),
    )
    .filter(pl.col("weight") >= min_directed_edge_weight)
    .sort("weight", descending=True)
)

write_frame(directed_edges, "network_directed_edges.csv")

directed_graph = nx.DiGraph()
for row in directed_edges.iter_rows(named=True):
    directed_graph.add_edge(row["source"], row["target"], weight=float(row["weight"]), emails=int(row["emails"]))
    directed_graph.nodes[row["source"]]["name"] = row["source_name"]
    directed_graph.nodes[row["target"]]["name"] = row["target_name"]

log("built directed graph")
print(f"directed nodes: {directed_graph.number_of_nodes():,}")
print(f"directed edges: {directed_graph.number_of_edges():,}")
display(directed_edges.head(20))

## Co-participation graph

An undirected edge links people who appear on the same email, regardless of sender/recipient role. Very crowded emails are skipped by default because they can turn mailing-list blasts into dense cliques.

In [ ]:
participant_sub = participants.select("email_id", "person_id", "person_name")
name_by_id = dict(zip(
    participant_sub.unique(subset=["person_id"])["person_id"].to_list(),
    participant_sub.unique(subset=["person_id"])["person_name"].to_list(),
))
edge_counts = Counter()
skipped_crowded_emails = 0

for email_id, group in participant_sub.group_by("email_id"):
    person_ids = sorted(group["person_id"].drop_nulls().unique().to_list())
    if len(person_ids) < 2:
        continue
    if len(person_ids) > max_people_per_email:
        skipped_crowded_emails += 1
        continue
    for left, right in combinations(person_ids, 2):
        edge_counts[(left, right)] += 1

copresence_edge_rows = [
    {
        "source": left,
        "source_name": name_by_id.get(left, left),
        "target": right,
        "target_name": name_by_id.get(right, right),
        "weight": weight,
    }
    for (left, right), weight in edge_counts.items()
    if weight >= min_copresence_edge_weight
]
copresence_edges = pl.DataFrame(copresence_edge_rows).sort("weight", descending=True)
write_frame(copresence_edges, "network_copresence_edges.csv")

copresence_graph = nx.Graph()
for row in copresence_edges.iter_rows(named=True):
    copresence_graph.add_edge(row["source"], row["target"], weight=float(row["weight"]))
    copresence_graph.nodes[row["source"]]["name"] = row["source_name"]
    copresence_graph.nodes[row["target"]]["name"] = row["target_name"]

log("built co-participation graph")
print(f"skipped crowded emails: {skipped_crowded_emails:,}")
print(f"co-participation nodes: {copresence_graph.number_of_nodes():,}")
print(f"co-participation edges: {copresence_graph.number_of_edges():,}")
display(copresence_edges.head(20))

## Network-level metrics

In [ ]:
def weighted_density(graph):
    if graph.number_of_nodes() < 2:
        return np.nan
    return nx.density(graph)


def largest_component_pct(graph):
    if graph.number_of_nodes() == 0:
        return np.nan
    if graph.is_directed():
        component_sizes = [len(component) for component in nx.weakly_connected_components(graph)]
    else:
        component_sizes = [len(component) for component in nx.connected_components(graph)]
    return max(component_sizes) / graph.number_of_nodes() * 100


def centralization(graph, centrality_function):
    if graph.number_of_nodes() < 3:
        return np.nan
    values = list(centrality_function(graph).values())
    observed = sum(max(values) - value for value in values)
    star = nx.star_graph(graph.number_of_nodes() - 1)
    star_values = list(centrality_function(star).values())
    denominator = sum(max(star_values) - value for value in star_values)
    if denominator == 0:
        return np.nan
    return observed / denominator


def approx_betweenness(graph):
    sample_size = min(centrality_sample_nodes, graph.number_of_nodes())
    if graph.number_of_nodes() <= centrality_sample_nodes:
        return nx.betweenness_centrality(graph, weight="distance", normalized=True)
    return nx.betweenness_centrality(graph, k=sample_size, seed=sample_seed, weight="distance", normalized=True)

for _, _, data in copresence_graph.edges(data=True):
    data["distance"] = 1 / data["weight"]
for _, _, data in directed_graph.edges(data=True):
    data["distance"] = 1 / data["weight"]

metrics = pl.DataFrame(
    [
        {
            "graph": "directed_sender_to_recipient",
            "nodes": directed_graph.number_of_nodes(),
            "edges": directed_graph.number_of_edges(),
            "density": weighted_density(directed_graph),
            "largest_component_pct": largest_component_pct(directed_graph),
        },
        {
            "graph": "undirected_email_copresence",
            "nodes": copresence_graph.number_of_nodes(),
            "edges": copresence_graph.number_of_edges(),
            "density": weighted_density(copresence_graph),
            "largest_component_pct": largest_component_pct(copresence_graph),
            "degree_centralization": centralization(copresence_graph, nx.degree_centrality),
            "betweenness_centralization": centralization(copresence_graph, approx_betweenness),
            "transitivity": nx.transitivity(copresence_graph) if copresence_graph.number_of_nodes() else np.nan,
        },
    ]
)
write_frame(metrics, "network_metrics.csv")
display(metrics)

## Central actors

In [ ]:
degree = dict(copresence_graph.degree(weight=None))
weighted_degree = dict(copresence_graph.degree(weight="weight"))
betweenness = approx_betweenness(copresence_graph)
closeness = nx.closeness_centrality(copresence_graph, distance="distance") if copresence_graph.number_of_nodes() else {}

directed_in = dict(directed_graph.in_degree(weight="weight"))
directed_out = dict(directed_graph.out_degree(weight="weight"))

centrality_rows = []
for person_id in copresence_graph.nodes:
    centrality_rows.append(
        {
            "person_id": person_id,
            "person_name": copresence_graph.nodes[person_id].get("name", person_id),
            "degree": degree.get(person_id, 0),
            "weighted_degree": weighted_degree.get(person_id, 0),
            "betweenness": betweenness.get(person_id, 0),
            "closeness": closeness.get(person_id, 0),
            "directed_in_weight": directed_in.get(person_id, 0),
            "directed_out_weight": directed_out.get(person_id, 0),
        }
    )
centrality = pl.DataFrame(centrality_rows).sort(
    ["betweenness", "weighted_degree", "degree"], descending=True
)
write_frame(centrality, "network_centrality.csv")
display(centrality.head(30))

plot_rows = centrality.head(25).reverse()
plt.figure(figsize=(9, 6))
plt.barh(plot_rows["person_name"], plot_rows["betweenness"], color="#4e79a7")
plt.xlabel("Approx. weighted betweenness")
plt.title("Top broker nodes in the email co-participation network")
plt.tight_layout()
plt.savefig(Path("outputs") / "network_top_betweenness.png", dpi=160)
plt.show()

## Communities

In [ ]:
if copresence_graph.number_of_edges():
    communities = list(nx.community.greedy_modularity_communities(copresence_graph, weight="weight"))
else:
    communities = []
community_by_person = {}
for community_index, community in enumerate(communities, start=1):
    for person_id in community:
        community_by_person[person_id] = community_index

community_rows = []
for community_index, community in enumerate(communities, start=1):
    members = sorted(
        (copresence_graph.nodes[person_id].get("name", person_id), person_id) for person_id in community
    )
    top_members = centrality.filter(pl.col("person_id").is_in(list(community))).head(10)
    community_rows.append(
        {
            "community": community_index,
            "members": len(community),
            "top_members_by_betweenness": "; ".join(top_members["person_name"].to_list()),
        }
    )
communities_table = pl.DataFrame(community_rows).sort("members", descending=True)
write_frame(communities_table, "network_communities.csv")
display(communities_table.head(20))

## Simulated disruption

This removes high-centrality actors from the co-participation graph and measures fragmentation. The paper reports 71-78% fragmentation after central-actor removal in individual-dominated criminal networks; the exact removal protocol is not visible from the abstract, so treat this as a structurally similar stress test rather than a direct replication.

In [ ]:
def disruption_rows(graph, ranked_nodes, label):
    rows = []
    original_nodes = graph.number_of_nodes()
    for remove_count in [1, 3, 5, 10, max(1, round(original_nodes * 0.05)), max(1, round(original_nodes * 0.10))]:
        selected = list(ranked_nodes)[:min(remove_count, original_nodes)]
        reduced = graph.copy()
        reduced.remove_nodes_from(selected)
        if reduced.number_of_nodes() == 0:
            largest = 0
            components = 0
        else:
            sizes = [len(component) for component in nx.connected_components(reduced)]
            largest = max(sizes)
            components = len(sizes)
        rows.append(
            {
                "strategy": label,
                "removed_nodes": len(selected),
                "removed_pct": len(selected) / original_nodes * 100 if original_nodes else np.nan,
                "largest_component_nodes": largest,
                "largest_component_pct_original": largest / original_nodes * 100 if original_nodes else np.nan,
                "fragmentation_pct_original": 100 - largest / original_nodes * 100 if original_nodes else np.nan,
                "component_count": components,
                "removed_people": "; ".join(copresence_graph.nodes[node].get("name", node) for node in selected[:15]),
            }
        )
    return rows

ranked_by_betweenness = centrality.sort("betweenness", descending=True)["person_id"].to_list()
ranked_by_degree = centrality.sort("weighted_degree", descending=True)["person_id"].to_list()
rows = disruption_rows(copresence_graph, ranked_by_betweenness, "remove_high_betweenness")
rows += disruption_rows(copresence_graph, ranked_by_degree, "remove_high_weighted_degree")

rng = np.random.default_rng(sample_seed)
all_nodes = np.array(list(copresence_graph.nodes))
for remove_count in [1, 3, 5, 10, max(1, round(copresence_graph.number_of_nodes() * 0.05)), max(1, round(copresence_graph.number_of_nodes() * 0.10))]:
    fragmentations = []
    for _ in range(random_disruption_runs):
        selected = rng.choice(all_nodes, size=min(remove_count, len(all_nodes)), replace=False)
        reduced = copresence_graph.copy()
        reduced.remove_nodes_from(selected)
        if reduced.number_of_nodes() == 0:
            fragmentations.append(100)
        else:
            largest = max(len(component) for component in nx.connected_components(reduced))
            fragmentations.append(100 - largest / copresence_graph.number_of_nodes() * 100)
    rows.append(
        {
            "strategy": "random_baseline_mean",
            "removed_nodes": min(remove_count, len(all_nodes)),
            "removed_pct": min(remove_count, len(all_nodes)) / len(all_nodes) * 100 if len(all_nodes) else np.nan,
            "largest_component_nodes": np.nan,
            "largest_component_pct_original": np.nan,
            "fragmentation_pct_original": float(np.mean(fragmentations)),
            "component_count": np.nan,
            "removed_people": "",
        }
    )

disruption = pl.DataFrame(rows).unique(subset=["strategy", "removed_nodes"]).sort(["removed_nodes", "strategy"])
write_frame(disruption, "network_disruption.csv")
display(disruption)

## Comparison to Pokorny 2026

Benchmarks below are from the public abstract/search result for Laszlo Pokorny's February 2026 work, which reports: Epstein network n = 84; criminal networks had mean degree centralization 0.70 vs. legitimate elite baseline 0.16, a 4.4x degree centralization ratio, a 7.7x betweenness centralization ratio, and 71-78% fragmentation after removing central actors.

Source found during notebook creation: [Laszlo Pokorny profile listing](https://njcu.academia.edu/LaszloPokorny) surfaced by exact-title search. Replace this with a full citation if you have the PDF or dissertation record.

In [ ]:
our_undirected = metrics.filter(pl.col("graph") == "undirected_email_copresence").row(0, named=True)
our_best_disruption = disruption.filter(pl.col("strategy").is_in(["remove_high_betweenness", "remove_high_weighted_degree"])).sort(
    "fragmentation_pct_original", descending=True
).head(1)

comparison = pl.DataFrame(
    [
        {
            "metric": "nodes",
            "our_email_network": our_undirected.get("nodes"),
            "pokorny_epstein_facilitation_network": 84,
            "interpretation": "Different data boundary: sampled email network vs public-source facilitation network.",
        },
        {
            "metric": "degree_centralization",
            "our_email_network": our_undirected.get("degree_centralization"),
            "pokorny_epstein_facilitation_network": np.nan,
            "pokorny_criminal_network_mean": 0.70,
            "pokorny_legitimate_baseline_mean": 0.16,
            "interpretation": "Higher values indicate concentration around a small set of actors.",
        },
        {
            "metric": "betweenness_centralization",
            "our_email_network": our_undirected.get("betweenness_centralization"),
            "pokorny_criminal_vs_legitimate_ratio": 7.7,
            "interpretation": "Use the ratio as a directional benchmark unless the full paper gives raw Epstein betweenness centralization.",
        },
        {
            "metric": "central_actor_fragmentation_pct",
            "our_email_network": float(our_best_disruption["fragmentation_pct_original"][0]) if our_best_disruption.height else np.nan,
            "pokorny_reported_range": "71-78",
            "interpretation": "Our value depends on the removal count/strategy table above; compare by matching the paper's protocol when available.",
        },
    ]
)
write_frame(comparison, "network_pokorny_comparison.csv")
display(comparison)

## Export metadata

In [ ]:
metadata = {
    "sample_documents": sample_documents,
    "sample_seed": sample_seed,
    "min_directed_edge_weight": min_directed_edge_weight,
    "min_copresence_edge_weight": min_copresence_edge_weight,
    "max_people_per_email": max_people_per_email,
    "centrality_sample_nodes": centrality_sample_nodes,
    "random_disruption_runs": random_disruption_runs,
    "linked_emails_available": analysis_emails.height,
    "sampled_linked_emails": sampled_emails.height,
    "directed_nodes": directed_graph.number_of_nodes(),
    "directed_edges": directed_graph.number_of_edges(),
    "copresence_nodes": copresence_graph.number_of_nodes(),
    "copresence_edges": copresence_graph.number_of_edges(),
    "runtime_seconds": round(perf_counter() - started_at, 2),
}
(Path("outputs") / "network_run_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(json.dumps(metadata, indent=2))